In [1]:
# ============================================================
# π² XOR Network — PyTorch Training Notebook
# ============================================================
# This notebook trains a 2-10-2 feedforward network using the
# π² (Processing-in-Interconnect) neuron model (MPLayer_in_K).
# The trained weights are quantized and exported as CSV files
# that drive the OMNeT++/INET TSN simulation.
#
# Flow: Train (MPLayer_in_K) → fine-tune γ → quantize (4-bit)
#       → export CSVs → use as PCP / timing offsets in OMNeT++
# ============================================================

import os
import pandas as pd
import seaborn as sn
import torch
from IPython.core.display import display
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
from sklearn.model_selection import train_test_split
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import StepLR
from torch.optim.lr_scheduler import _LRScheduler
import math
device = "cuda"


In [2]:
# import torch
# from torch.autograd import gradcheck

# # Create dummy input
# batch_size, in_size, out_size = 2, 5, 3  # small sizes for gradcheck
# gamma = 2

# # Define dummy input for gradcheck
# input_tensor = torch.randn(batch_size, in_size, out_size, dtype=torch.float64, requires_grad=True)

# # Define wrapper function for gradcheck
# class SpikeMPK(torch.autograd.Function):
#     @staticmethod
#     def forward(ctx, inMat1, gamma):
#         ctx.gamma = gamma
#         ctx.save_for_backward(inMat1)

#         if gamma == 0 or gamma == 1:
#             out = torch.kthvalue(inMat1, 1, dim=1).values
#             return out
#         thr, _ = torch.topk(inMat1, int(gamma), dim=1, largest=False, sorted=False)
#         sum_nonzero = thr.sum(dim=1)
#         return sum_nonzero / gamma  # average of the smallest gamma values

#     @staticmethod
#     def backward(ctx, grad_output):
#         gamma = ctx.gamma
#         inMat1, = ctx.saved_tensors
#         grad_input = torch.zeros_like(inMat1)
#         thr,indices = torch.topk(inMat1, gamma, dim=1, largest=False, sorted=False)
#         grad_vals = (grad_output / gamma).unsqueeze(1).expand_as(indices)
#         grad_input.scatter_(dim=1, index=indices, src=grad_vals)
#         return grad_input, None

# # Wrap it for gradcheck (inputs must be tuple)
# input_for_gradcheck = (input_tensor, gamma)

# # Run gradcheck
# print("Running gradcheck...")
# test_result = gradcheck(SpikeMPK.apply, input_for_gradcheck, eps=1e-6, atol=1e-4, rtol=1e-2)
# print("Gradcheck passed:", test_result)


In [3]:
# ============================================================
# MPLayer_in_K — π² Neuron Layer
# ============================================================
# Implements one layer of the π²-mesh network.
#
# Key idea:
#   In the π² model, a neuron fires at time T = min_k(x_i + |w_ij|),
#   where x_i is the input spike arrival time and w_ij is the weight.
#   'gamma' (γ) is the order statistic — γ=1 takes the earliest
#   arriving spike; γ>1 averages the γ earliest, smoothing the output.
#
# Differential encoding:
#   Inputs and weights are split into positive (⁺) and negative (⁻)
#   copies so that signed arithmetic maps to spike-time differences:
#     z⁺ = SpikeK(x⁺ + W⁺, x⁻ + W⁻)   (positive path)
#     z⁻ = SpikeK(x⁺ + W⁻, x⁻ + W⁺)   (negative path)
#   Output = z⁺ − z⁻  (or both returned separately when diff=1)
#
# OMNeT++ mapping:
#   x⁺/x⁻ → initialProductionOffset of xp/xm packets (inn[*])
#   W⁺/W⁻ → PCP priority value in stream-coder encoder mappings
#   SpikeK → ATS EligibilityTimeMeter (maxResidenceTime ladder)
#   ReLU   → CBS CreditBasedGate
# ============================================================

class MPLayer_in_K(torch.nn.Module):
  def __init__(self, inp_node, out_node, gamma, diff=0):
    """
    Args:
        inp_node: number of input neurons
        out_node: number of output neurons
        gamma:    order statistic for SpikeK (1 = min; >1 = avg of γ smallest)
        diff:     0 → return scalar z⁺−z⁻; 1 → return (z⁺, z⁻) separately
    """
    super().__init__()
    self.inp_node = inp_node
    self.out_node = out_node
    self.gamma = gamma
    self.diff = diff  # whether to return differential outputs
    torch.manual_seed(43)
    self.weight = torch.nn.Parameter(torch.empty(inp_node, out_node), requires_grad=True)
    torch.nn.init.xavier_normal_(self.weight, gain=1.0)
    torch.clamp(self.weight, -3, 3)
    self.fn = MPLayer_in_K.SpikeK.apply

  def forward(self, inputp, inputn=None):
    """
    Forward pass.
    If inputn is None, the layer generates x⁺ = 1+x and x⁻ = 1-x
    internally (single-ended input mode).
    If inputn is provided, both positive and negative spike times are
    passed explicitly (differential input mode — used between layers).
    """
    inputp = torch.unsqueeze(inputp, axis=-1)
    self.weight.type_as(inputp)

    if inputn is None:
        # Single-ended: generate differential encoding from scalar input
        plusIn  = (1 + inputp)   # x⁺: early arrival → large positive value
        minusIn = (1 - inputp)   # x⁻: complementary spike time
    else:
        minusIn = torch.unsqueeze(inputn, axis=-1)
        plusIn  = inputp

    # Decompose weights into excitatory (W⁺) and inhibitory (W⁻) parts
    plusW  = F.relu(self.weight)   # W⁺ = max(W, 0)
    minusW = F.relu(-self.weight)  # W⁻ = max(-W, 0)

    # Compute positive and negative spike-time sums (arrival time + weight delay)
    zPlus  = torch.cat([(plusIn + plusW),  (minusIn + minusW)], axis=1)  # excitatory path
    zMinus = torch.cat([(plusIn + minusW), (minusIn + plusW)],  axis=1)  # inhibitory path

    # Apply γ-order statistic (SpikeK): neuron fires at the γ-th earliest input
    zPlus  = self.fn(zPlus,  self.gamma)
    zMinus = self.fn(zMinus, self.gamma)
    torch.cuda.empty_cache()

    if self.diff == 0:
        return zPlus - zMinus  # scalar output: net spike-time difference
    else:
        return zPlus, zMinus   # differential output: pass both to next layer

  @staticmethod
  class SpikeK(torch.autograd.Function):
    """
    Custom autograd function implementing the γ-order-statistic neuron.

    Forward:  returns the average of the γ smallest values along dim=1.
              γ=1 → minimum (first-to-fire); γ>1 → soft threshold.
    Backward: gradient flows only through the γ inputs that were selected
              (sparse gradient, similar to top-k backward).

    In the TSN simulation this is implemented by the ATS switch:
    the EligibilityTimeMeter's maxResidenceTime encodes the weight delay,
    and the γ-th earliest-arriving packet wins (fires the neuron).
    """
    @staticmethod
    def forward(ctx, inMat1, gamma):
        ctx.gamma = gamma
        ctx.save_for_backward(inMat1)
        if gamma == 0 or gamma == 1:
            # γ=1: first spike fires the neuron (minimum arrival time)
            out = torch.kthvalue(inMat1, 1, dim=1).values
            return out
        # γ>1: average of the γ smallest (softer integration)
        thr, _ = torch.topk(inMat1, int(gamma), dim=1, largest=False, sorted=False)
        sum_nonzero = thr.sum(dim=1)
        return sum_nonzero / gamma

    @staticmethod
    def backward(ctx, grad_output):
        # Gradient flows only through the γ selected (earliest) inputs
        gamma = ctx.gamma
        inMat1, = ctx.saved_tensors
        grad_input = torch.zeros_like(inMat1)
        thr, indices = torch.topk(inMat1, gamma, dim=1, largest=False, sorted=False)
        grad_vals = (grad_output / gamma).unsqueeze(1).expand_as(indices)
        grad_input.scatter_(dim=1, index=indices, src=grad_vals)
        return grad_input, None


In [4]:
# import torch
# import torch.nn as nn

# # Assume MPLayer_in_K is already defined exactly as you posted

# # Create input
# batch_size = 1
# input_dim = 2
# output_dim = 10
# gamma = 1

# # Instantiate the layer
# layer = MPLayer_in_K(inp_node=input_dim, out_node=output_dim, gamma=gamma, diff=0)
# print("wts")
# # layer.weight.data = quant_k1.clone()
# print("wts")
# print(layer.weight.data)
# # Create random input tensor
# input_tensor = torch.randn(batch_size, input_dim)
# input_tensor = torch.tensor([[ 0.2392, -0.2890]])
# # Forward pass (non-differential input)
# output = layer(input_tensor.clone())

# print("Input:")
# print(input_tensor)
# print("\nOutput:")
# print(output)
# print("\nOutput shape:", output.shape)


In [5]:
# ============================================================
# Dataset Generation and Initial Training (γ = [2, 5])
# ============================================================
# XOR dataset: 2000 samples uniform on [-1,1]².
# Label Y = XOR(x0>0, x1>0) — four quadrant binary classification.
#
# Initial γ values are deliberately high (K=[2,5]) so the network
# has a smooth gradient landscape to start from. γ is progressively
# reduced in subsequent cells to approach the hardware target (γ=1).
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import shuffle

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Parameters
N = 2000        # total samples
input_dim  = 2  # two XOR inputs
hidden_dim = 10 # hidden neurons (maps to hidd[0..9] in OMNeT++)
output_dim = 2  # class0 / class1
lr = 0.01
epochs = 20

# XOR-style data: uniform inputs, label = sign(x0) XOR sign(x1)
X = np.random.uniform(low=-1, high=1, size=(N, 2))
Y = np.bitwise_xor(X[:, 0] > 0, X[:, 1] > 0).astype(int)
X, Y = shuffle(X, Y)

X_t = torch.from_numpy(X).float()
Y_t = torch.from_numpy(Y).long()

# 80/20 split
X_tensor, X_test, Y_tensor, Y_test = train_test_split(X_t, Y_t, test_size=0.2, random_state=42)
N, M = X_tensor.shape

# Stage 1: train with γ=[2,5] — smooth landscape, fast convergence
K = [2, 5]
layer1 = MPLayer_in_K(inp_node=input_dim, out_node=hidden_dim, gamma=K[0], diff=0)
layer2 = MPLayer_in_K(inp_node=hidden_dim, out_node=output_dim, gamma=K[1], diff=0)

optimizer = torch.optim.SGD(list(layer1.parameters()) + list(layer2.parameters()), lr=lr)

for epoch in range(epochs):
    total_loss = 0.0
    for i in range(N):
        x_i = X_tensor[i].view(1, -1)
        y_i = Y_tensor[i].view(-1)

        optimizer.zero_grad()
        z1 = layer1(x_i)
        a1 = F.relu(z1)       # ReLU activation (→ CBS gate in OMNeT++)
        z2 = layer2(a1)

        loss = F.cross_entropy(z2 * 10, y_i)  # *10 sharpens the softmax
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Avg Loss: {total_loss / N:.4f}")
    with torch.no_grad():
        Z1 = F.relu(layer1(X_tensor))
        Z2 = layer2(Z1)
        preds = torch.argmax(Z2, dim=1)
        accuracy = (preds == Y_tensor).float().mean().item()
        print(f"Final Classification Accuracy: {accuracy * 100:.2f}%")

# Test accuracy
with torch.no_grad():
    Z1 = F.relu(layer1(X_test))
    Z2 = layer2(Z1)
    preds = torch.argmax(Z2, dim=1)
    accuracy = (preds == Y_test).float().mean().item()
    print(f"test Classification Accuracy: {accuracy * 100:.2f}%")


Epoch 1, Avg Loss: 0.2060
Final Classification Accuracy: 99.37%
Epoch 2, Avg Loss: 0.1117
Final Classification Accuracy: 99.87%
Epoch 3, Avg Loss: 0.1007
Final Classification Accuracy: 99.75%
Epoch 4, Avg Loss: 0.0988
Final Classification Accuracy: 99.75%
Epoch 5, Avg Loss: 0.0980
Final Classification Accuracy: 99.81%
Epoch 6, Avg Loss: 0.0976
Final Classification Accuracy: 99.75%
Epoch 7, Avg Loss: 0.0973
Final Classification Accuracy: 99.75%
Epoch 8, Avg Loss: 0.0971
Final Classification Accuracy: 99.69%
Epoch 9, Avg Loss: 0.0970
Final Classification Accuracy: 99.62%
Epoch 10, Avg Loss: 0.0968
Final Classification Accuracy: 99.62%
Epoch 11, Avg Loss: 0.0967
Final Classification Accuracy: 99.62%
Epoch 12, Avg Loss: 0.0966
Final Classification Accuracy: 99.62%
Epoch 13, Avg Loss: 0.0964
Final Classification Accuracy: 99.69%
Epoch 14, Avg Loss: 0.0963
Final Classification Accuracy: 99.69%
Epoch 15, Avg Loss: 0.0962
Final Classification Accuracy: 99.62%
Epoch 16, Avg Loss: 0.0962
Final C

In [6]:
# ============================================================
# Stage 2: Fine-tune with γ = [1, 2]
# ============================================================
# Reduce γ progressively toward hardware target (γ=1).
# The weights from Stage 1 are reused — only γ changes.
# A lower γ tightens the order-statistic and moves closer to
# the first-spike-fires rule used by the ATS switch.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import shuffle

torch.manual_seed(42)
np.random.seed(42)

lr = 0.001
epochs = 10

# Transfer weights from Stage 1; reduce γ to [1, 2]
K = [1, 2]
W1 = layer1.weight.data
W2 = layer2.weight.data
layer11 = MPLayer_in_K(inp_node=input_dim, out_node=hidden_dim, gamma=K[0], diff=0)
layer22 = MPLayer_in_K(inp_node=hidden_dim, out_node=output_dim, gamma=K[1], diff=0)
layer11.weight.data = W1.clone()
layer22.weight.data = W2.clone()

# Baseline accuracy before fine-tuning (should match Stage 1 test accuracy)
with torch.no_grad():
    Z1 = F.relu(layer11(X_tensor))
    Z2 = layer22(Z1)
    preds = torch.argmax(Z2, dim=1)
    accuracy = (preds == Y_tensor).float().mean().item()
    print(f"Final Classification Accuracy: {accuracy * 100:.2f}%")

optimizer = torch.optim.SGD(list(layer11.parameters()) + list(layer22.parameters()), lr=lr)

for epoch in range(epochs):
    total_loss = 0.0
    for i in range(N):
        x_i = X_tensor[i].view(1, -1)
        y_i = Y_tensor[i].view(-1)

        optimizer.zero_grad()
        z1 = layer11(x_i)
        a1 = F.relu(z1)
        z2 = layer22(a1)

        loss = F.cross_entropy(z2 * 10, y_i)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Avg Loss: {total_loss / N:.4f}")
    with torch.no_grad():
        Z1 = F.relu(layer11(X_tensor))
        Z2 = layer22(Z1)
        preds = torch.argmax(Z2, dim=1)
        accuracy = (preds == Y_tensor).float().mean().item()
        print(f"Final Classification Accuracy: {accuracy * 100:.2f}%")

with torch.no_grad():
    Z1 = F.relu(layer11(X_test))
    Z2 = layer22(Z1)
    preds = torch.argmax(Z2, dim=1)
    accuracy = (preds == Y_test).float().mean().item()
    print(f"test Classification Accuracy: {accuracy * 100:.2f}%")


Final Classification Accuracy: 96.88%
Epoch 1, Avg Loss: 0.0892
Final Classification Accuracy: 98.44%
Epoch 2, Avg Loss: 0.0768
Final Classification Accuracy: 99.06%
Epoch 3, Avg Loss: 0.0681
Final Classification Accuracy: 99.37%
Epoch 4, Avg Loss: 0.0626
Final Classification Accuracy: 99.56%
Epoch 5, Avg Loss: 0.0585
Final Classification Accuracy: 99.62%
Epoch 6, Avg Loss: 0.0555
Final Classification Accuracy: 99.69%
Epoch 7, Avg Loss: 0.0532
Final Classification Accuracy: 99.87%
Epoch 8, Avg Loss: 0.0514
Final Classification Accuracy: 99.81%
Epoch 9, Avg Loss: 0.0501
Final Classification Accuracy: 99.81%
Epoch 10, Avg Loss: 0.0492
Final Classification Accuracy: 99.87%
test Classification Accuracy: 99.50%


In [7]:
# ============================================================
# Stage 3: Fine-tune with γ = [1, 1] — hardware-exact model
# ============================================================
# γ=1 in both layers means the neuron fires at the EARLIEST
# arriving spike — exactly the first-spike-fires rule implemented
# by the ATS EligibilityTimeMeter in OMNeT++.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import shuffle

torch.manual_seed(42)
np.random.seed(42)

# Transfer weights from Stage 2; set γ=[1,1]
K = [1, 1]
W1 = layer11.weight.data
W2 = layer22.weight.data
layer111 = MPLayer_in_K(inp_node=input_dim, out_node=hidden_dim, gamma=K[0], diff=0)
layer222 = MPLayer_in_K(inp_node=hidden_dim, out_node=output_dim, gamma=K[1], diff=0)
layer111.weight.data = W1.clone()
layer222.weight.data = W2.clone()

with torch.no_grad():
    Z1 = F.relu(layer111(X_tensor))
    Z2 = layer222(Z1)
    preds = torch.argmax(Z2, dim=1)
    accuracy = (preds == Y_tensor).float().mean().item()
    print(f"Final Classification Accuracy: {accuracy * 100:.2f}%")

optimizer = torch.optim.SGD(list(layer111.parameters()) + list(layer222.parameters()), lr=lr)
epochs = 1
lr = 0.001

for epoch in range(epochs):
    total_loss = 0.0
    for i in range(N):
        x_i = X_tensor[i].view(1, -1)
        y_i = Y_tensor[i].view(-1)

        optimizer.zero_grad()
        z1 = layer111(x_i)
        a1 = F.relu(z1)
        z2 = layer222(a1)

        loss = F.cross_entropy(z2 * 10, y_i)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Avg Loss: {total_loss / N:.4f}")
    with torch.no_grad():
        Z1 = F.relu(layer111(X_tensor))
        Z2 = layer222(Z1)
        preds = torch.argmax(Z2, dim=1)
        accuracy = (preds == Y_tensor).float().mean().item()
        print(f"Final Classification Accuracy: {accuracy * 100:.2f}%")

with torch.no_grad():
    Z1 = F.relu(layer111(X_test))
    Z2 = layer222(Z1)
    preds = torch.argmax(Z2, dim=1)
    accuracy = (preds == Y_test).float().mean().item()
    print(f"test Classification Accuracy: {accuracy * 100:.2f}%")


Final Classification Accuracy: 99.87%
Epoch 1, Avg Loss: 0.0429
Final Classification Accuracy: 100.00%
test Classification Accuracy: 99.75%


In [8]:
# ============================================================
# Verify hardware-exact inference (γ=[1,1], full-precision weights)
# ============================================================
# Quick accuracy check before quantization to confirm the
# γ=[1,1] model still performs well on the full dataset.
# ============================================================

# Define model layers using nn.Linear
K = [1, 1]
W1 = layer111.weight.data
W2 = layer222.weight.data
layer1k = MPLayer_in_K(inp_node=input_dim, out_node=hidden_dim, gamma=K[0], diff=0)
layer2k = MPLayer_in_K(inp_node=hidden_dim, out_node=output_dim, gamma=K[1], diff=0)
layer1k.weight.data = W1.clone()
layer2k.weight.data = W2.clone()

with torch.no_grad():
    Z1 = F.relu(layer1k(X_tensor))
    Z2 = layer2k(Z1)
    preds = torch.argmax(Z2, dim=1)
    accuracy = (preds == Y_tensor).float().mean().item()
    print(f"Final Classification Accuracy: {accuracy * 100:.2f}%")


Final Classification Accuracy: 100.00%


In [9]:
# ============================================================
# 4-bit Symmetric Quantization and Weight Export
# ============================================================
# The π² TSN simulation encodes weights as PCP values (0–7),
# so weights must be mapped to small integers.
#
# Scheme:
#   n = 4 bits → q = 2^(n-1) - 1 = 7
#   A shared scale factor is computed across W1 and W2:
#     scale = q / max(|W1|, |W2|)
#   Integer weights: int_k = round(W * scale)
#   Dequantized:     quant_k = int_k / scale
#
# Outputs written to:
#   w1_xor.csv  — integer W1 (2×10), used to set PCP in omnetpp_w1_*.ini
#   w2_xor.csv  — integer W2 (10×2), used to set PCP in omnetpp_w2_*.ini
# ============================================================

W1k = layer1k.weight.data
W2k = layer2k.weight.data
print(W1k)
print(W2k)

n = 4                          # number of quantization bits
q = 2**(n-1) - 1               # max integer value = 7
print(q)

# Shared scale across both weight matrices (keeps relative magnitudes)
k1 = torch.max(torch.abs(W1k))
k2 = torch.max(torch.abs(W2k))
max_val = torch.max(k1, k2)
scaling_factor = q / max_val
print(scaling_factor)

int_k1  = torch.round(W1k * scaling_factor)   # integer weights for layer 1
int_k2  = torch.round(W2k * scaling_factor)   # integer weights for layer 2
quant_k1 = int_k1 / scaling_factor            # dequantized (for accuracy check)
quant_k2 = int_k2 / scaling_factor

# Accuracy check with quantized weights
K = [1, 1]
layer1k = MPLayer_in_K(inp_node=input_dim, out_node=hidden_dim, gamma=K[0], diff=0)
layer2k = MPLayer_in_K(inp_node=hidden_dim, out_node=output_dim, gamma=K[1], diff=0)
layer1k.weight.data = quant_k1.clone()
layer2k.weight.data = quant_k2.clone()

with torch.no_grad():
    Z1 = F.relu(layer1k(X_tensor))
    Z2 = layer2k(Z1)
    preds = torch.argmax(Z2, dim=1)
    accuracy = (preds == Y_tensor).float().mean().item()
    print(f"Final Classification Accuracy: {accuracy * 100:.2f}%")

# Differential-mode test (matches OMNeT++ forward pass exactly):
#   layer1k receives x⁺=1+x and x⁻=1-x explicitly
with torch.no_grad():
    Z1 = F.relu(layer1k(1 + X_test, 1 - X_test))
    print(torch.max(Z1))
    Z2 = layer2k(Z1)
    preds = torch.argmax(Z2, dim=1)
    accuracy = (preds == Y_test).float().mean().item()
    print(f"Test Classification Accuracy: {accuracy * 100:.2f}%")

# Export integer weights as CSV (used directly in omnetpp_w*.ini PCP mappings)
filename = 'w1_xor.csv'
print(int_k1.size())
np.savetxt(filename, int_k1.cpu().numpy(), delimiter=',')

filename = 'w2_xor.csv'
print(int_k2.size())
np.savetxt(filename, int_k2.cpu().numpy(), delimiter=',')


tensor([[-0.8192, -0.1811,  0.9055,  0.9077,  0.9666, -1.0421, -0.7796, -1.0645,
         -0.0730,  1.0434],
        [ 0.7869,  0.0040, -0.8661,  0.8175, -0.9309,  1.0075, -0.7912, -0.9978,
         -0.2154,  1.0366]])
tensor([[ 0.7030, -0.7005],
        [ 0.8536,  0.2510],
        [ 0.7348, -0.7348],
        [-0.7067,  0.7067],
        [ 0.7411, -0.7422],
        [ 0.8010, -0.8008],
        [-0.6809,  0.6810],
        [-0.7729,  0.7729],
        [-0.1948, -0.7467],
        [-0.7803,  0.7803]])
7
tensor(6.5757)
Final Classification Accuracy: 99.94%
tensor(1.0645)
Test Classification Accuracy: 99.75%
torch.Size([2, 10])
torch.Size([10, 2])


In [10]:
# Print integer weights for inspection
# int_k1: shape (2, 10) — W1 from inputs to hidden neurons
# int_k2: shape (10, 2) — W2 from hidden neurons to output classes
print(int_k1)
print(int_k2)


tensor([[-5., -1.,  6.,  6.,  6., -7., -5., -7., -0.,  7.],
        [ 5.,  0., -6.,  5., -6.,  7., -5., -7., -1.,  7.]])
tensor([[ 5., -5.],
        [ 6.,  2.],
        [ 5., -5.],
        [-5.,  5.],
        [ 5., -5.],
        [ 5., -5.],
        [-4.,  4.],
        [-5.,  5.],
        [-1., -5.],
        [-5.,  5.]])


In [11]:
# ============================================================
# Differential Forward Pass Trace — OMNeT++ Verification
# ============================================================
# Runs the quantized model in differential mode (diff=1) to
# extract per-layer spike times (hp, hn, op, on) for each test case.
#
# These values are the ground truth for the OMNeT++ simulation:
#   - plusIn / minusIn  → initialProductionOffset in omnetpp_input_*.ini
#   - h1p / h1n         → expected spike times at hidden layer output
#   - o1p / o1n         → expected spike times at output layer
#
# The +2.6s and +4s offsets shift spike times into the positive domain
# so they can be encoded as packet transmission delays in OMNeT++.
# ============================================================

num_cases = 2   # number of XOR test cases to trace
correct = 0
total = 0

# Differential layers (diff=1 returns z⁺, z⁻ separately)
layer11k = MPLayer_in_K(inp_node=input_dim,  out_node=hidden_dim,  gamma=K[0], diff=1)
layer22k = MPLayer_in_K(inp_node=hidden_dim, out_node=output_dim,  gamma=K[1], diff=1)
layer11k.weight.data = quant_k1.clone()
layer22k.weight.data = quant_k2.clone()

print(torch.max(torch.abs(X_test)))    # input magnitude
print(torch.max(torch.abs(quant_k1)))  # W1 magnitude
print(torch.max(torch.abs(quant_k2)))  # W2 magnitude

with torch.no_grad():
    for i in range(num_cases):
        x_i = X_test[i].view(1, -1)
        print("input_tensor")
        print(x_i)
        y_i = Y_test[i].view(-1)

        # Differential encoding of input (x⁺ = 1+x, x⁻ = 1-x)
        plusIn  = (1 + x_i)   # early spike for positive input
        minusIn = (1 - x_i)   # complementary spike
        print("INPUTS")
        print(plusIn)
        print(minusIn)

        # Hidden layer: get positive and negative spike-time outputs
        hp, hn = layer11k(plusIn, minusIn)
        h = F.relu(hp - hn)   # ReLU activation (→ CBS gate in OMNeT++)
        print('hp'); print(hp)
        print('hn'); print(hn)

        # Shift hidden output into positive time domain for OMNeT++ encoding
        # +2.6s offset matches the startTime of hidd[*].app in omnetpp.ini
        h1p = 2.6 + h
        h1n = 2.6 - h
        print('hp-final'); print(h1p)
        print('hn-final'); print(h1n)

        # Output layer
        minusIn = torch.unsqueeze(h1n, axis=-1)
        plusIn  = torch.unsqueeze(h1p, axis=-1)
        print(W2k.size())
        print(plusIn  + F.relu(quant_k2))
        print(minusIn + F.relu(-quant_k2))

        op, on = layer22k(h1p, h1n)
        print('output'); print(op); print(on)

        # Shift output into positive domain (+4s matches outt[*].app startTime)
        o1p = 4 + F.relu(op - on)
        o1n = 4 - F.relu(op - on)
        print('o1p'); print(o1p)
        print('o1n'); print(o1n)
        print('y_i'); print(y_i)

# Export spike timing inputs for OMNeT++ (omnetpp_input_*.ini)
# plusin / minusIn shape: (num_cases, 2) — one row per test case
filename = 'plusin_xor.csv'
plusIn = 1 + X_test[0:num_cases].reshape(num_cases, 2)
print(plusIn.size())
np.savetxt(filename, plusIn.cpu().numpy(), delimiter=',')

filename = 'minusIn_xor.csv'
minusIn = 1 - X_test[0:num_cases].reshape(num_cases, 2)
print(minusIn.size())
np.savetxt(filename, minusIn.cpu().numpy(), delimiter=',')


tensor(1.0000)
tensor(1.0645)
tensor(0.9124)
input_tensor
tensor([[ 0.2392, -0.2890]])
INPUTS
tensor([[1.2392, 0.7110]])
tensor([[0.7608, 1.2890]])
hp
tensor([[1.2392, 0.7110, 0.7110, 0.7608, 0.7110, 1.2392, 0.7110, 0.7110, 0.7110,
         0.7608]])
hn
tensor([[0.7110, 0.7110, 1.2392, 0.7110, 1.2392, 0.7110, 0.7608, 0.7608, 0.7608,
         0.7110]])
hp-final
tensor([[3.1282, 2.6000, 2.6000, 2.6498, 2.6000, 3.1282, 2.6000, 2.6000, 2.6000,
         2.6498]])
hn-final
tensor([[2.0718, 2.6000, 2.6000, 2.5502, 2.6000, 2.0718, 2.6000, 2.6000, 2.6000,
         2.5502]])
torch.Size([10, 2])
tensor([[[3.8886, 3.1282],
         [3.5124, 2.9041],
         [3.3604, 2.6000],
         [2.6498, 3.4102],
         [3.3604, 2.6000],
         [3.8886, 3.1282],
         [2.6000, 3.2083],
         [2.6000, 3.3604],
         [2.6000, 2.6000],
         [2.6498, 3.4102]]])
tensor([[[2.0718, 2.8322],
         [2.6000, 2.6000],
         [2.6000, 3.3604],
         [3.3105, 2.5502],
         [2.6000, 3.3604],
 